In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



In [22]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

In [23]:
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [31]:


df = pd.read_csv("final_fraud_predictions.csv")
print("Dataset: ")
print(df.head())

Dataset: 
   Actual_Class  Predicted_Class  Fraud_Probability
0             0                0           0.030400
1             0                0           0.118886
2             0                0           0.045525
3             0                0           0.015312
4             0                0           0.048122


In [32]:
print("\nDataset Shape")
print(df.shape)


Dataset Shape
(118108, 3)


In [33]:
print("\nAll Feature Names:")
print(df.columns.tolist())
print("\nNumber of Features:")
print(len(df.columns))
print("\nFeature Information:")
print(df.info())


All Feature Names:
['Actual_Class', 'Predicted_Class', 'Fraud_Probability']

Number of Features:
3

Feature Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118108 entries, 0 to 118107
Data columns (total 3 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Actual_Class       118108 non-null  int64  
 1   Predicted_Class    118108 non-null  int64  
 2   Fraud_Probability  118108 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 2.7 MB
None


In [34]:
#Removing duplicates
print("\nDuplicates : " , df.duplicated().sum())


Duplicates :  6488


In [35]:
# CHECK FRAUD DISTRIBUTION

print("\nClass Distribution:")
print(df["Actual_Class"].value_counts())

print("\nClass Percentage:")
print(df["Actual_Class"].value_counts(normalize=True) * 100)


Class Distribution:
Actual_Class
0    113975
1      4133
Name: count, dtype: int64

Class Percentage:
Actual_Class
0    96.50066
1     3.49934
Name: proportion, dtype: float64


In [ ]:
#KEEP NUMERICAL FEATURES
x = x.select_dtypes(
    include=np.number
)
print("\nNumber of numerical features:")
print(x.shape[1])

In [ ]:
#SEPARATE FEATURES AND TARGET
X = df.drop(
    columns=["isFraud", "TransactionID"]
)
y = df["isFraud"]

In [ ]:
#HANDLE MISSING VALUES
print("\nTotal missing values before:")
print(X.isnull().sum().sum())
# Replace missing values with median
X = X.fillna(X.median())
print("\nTotal missing values after:")
print(X.isnull().sum().sum())

In [ ]:
#TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print("\nTraining Data Shape:")
print(X_train.shape)
print("\nTesting Data Shape:")
print(X_test.shape)
print("\nTraining Class Distribution:")
print(y_train.value_counts())

In [ ]:
#SCALE FEATURES
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(
    X_train
)
X_test_scaled = scaler.transform(
    X_test
)

In [ ]:
#APPLY SMOTE
smote = SMOTE(
    sampling_strategy=0.5,
    random_state=42
)
X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)
print("\nClass Distribution After SMOTE:")
print(
    pd.Series(y_train_smote).value_counts()
)

In [ ]:
#TRAIN XGBOOST
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)
print("\nTraining XGBoost...")
xgb_model.fit(
    X_train_smote,
    y_train_smote
)
print("Training Completed.")

In [ ]:
#GET FRAUD PROBABILITIES
y_probability = xgb_model.predict_proba(
    X_test_scaled
)[:, 1]
print("\nFirst 10 Fraud Probabilities:")
print(y_probability[:10])

In [ ]:
#DEFAULT THRESHOLD = 0.5
y_pred_05 = (
    y_probability >= 0.5
).astype(int)
print("\nXGBoost Results - Threshold 0.5")
print(
    classification_report(
        y_test,
        y_pred_05,
        zero_division=0
    )
)
print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        y_probability
    )
)
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_05
    )
)

In [ ]:
#TUNE DECISION THRESHOLD
thresholds = np.arange(
    0.1,
    0.91,
    0.05
)
results = []
for threshold in thresholds:

    y_pred = (
        y_probability >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    results.append(
        [threshold, precision, recall, f1]
    )
results_df = pd.DataFrame(
    results,
    columns=[
        "Threshold",
        "Precision",
        "Recall",
        "F1"
    ]
)
print("\nThreshold Results:")
print(results_df)

In [ ]:
#BEST THRESHOLD
best_row = results_df.loc[
    results_df["F1"].idxmax()
]
best_threshold = best_row["Threshold"]
print("\nBest Threshold:")
print(best_threshold)
print("\nBest F1 Score:")
print(best_row["F1"])

In [ ]:
#FINAL PREDICTION
y_pred = (
    y_probability >= best_threshold
).astype(int)

In [ ]:
# FINAL EVALUATION
print("\nFinal XGBoost Results:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)
print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)
print(
    "\nROC-AUC:",
    roc_auc_score(
        y_test,
        y_probability
    )
)
print(
    "Precision:",
    precision_score(
        y_test,
        y_pred,
        zero_division=0
    )
)
print(
    "Recall:",
    recall_score(
        y_test,
        y_pred,
        zero_division=0
    )
)
print(
    "F1 Score:",
    f1_score(
        y_test,
        y_pred,
        zero_division=0
    )
)

In [ ]:
#FEATURE IMPORTANCE
importance = pd.Series(
    xgb_model.feature_importances_,
    index=X.columns
)

importance = importance.sort_values(
    ascending=False
)
print("\nTop 10 Important Features:")
print(
    importance.head(10)
)

In [ ]:
#Feature Ploting
top_features = importance.head(10)

plt.figure(figsize=(9, 6))

top_features.sort_values().plot(
    kind="barh"
)
plt.title(
    "Top 10 XGBoost Feature Importance"
)
plt.xlabel(
    "Importance Score"
)
plt.ylabel(
    "Features"
)
plt.tight_layout()
plt.savefig(
    "top_features.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
#Saving Final Predictions
prediction_df = pd.DataFrame({
    "Actual_Class": y_test.values,
    "Predicted_Class": y_pred,
    "Fraud_Probability": y_probability
})
prediction_df.to_csv(
    "final_fraud_predictions.csv",
    index=False
)
print("Final predictions saved successfully!")
print("File: final_fraud_predictions.csv")
display(prediction_df.head())